# 📈 Tägliche Aktienanalyse — Jupyter Notebook

**LLM-gestützte technische Aktienanalyse mit Trend-Trading**

Dieses Notebook demonstriert die Kernfunktionen der Aktienanalyse:
- 📡 Kursdaten laden (Demo-Daten & Alpha Vantage)
- 📊 Technische Indikatoren (MA5/MA10/MA20, MACD, RSI)
- 🎯 Trend-Trading-Signale (Buy/Sell/Hold)
- 📈 Charts & Visualisierungen

---

## 1. Setup & Importe

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Repository-Pfad hinzufügen
sys.path.insert(0, '/opt/data/taegliche-aktienanalyse')

# Versuche, den echten Analyzer zu importieren
try:
    from src.stock_analyzer import StockTrendAnalyzer, TrendAnalysisResult, TrendStatus, BuySignal
    print('✅ StockTrendAnalyzer importiert')
except ImportError as e:
    print(f'⚠️ Import fehlgeschlagen: {e}')
    print('Verwende eigenständige Implementierung im Notebook.')

print(f'📅 Ausführungszeit: {datetime.now().strftime("%d.%m.%Y %H:%M")}')
print(f'🐍 Python: {sys.version}')

## 2. Demo-Kursdaten generieren

Erzeugt realistische OHLCV-Daten für eine Beispiel-Aktie (z.B. AAPL).

In [ ]:
def generate_stock_data(symbol: str = 'AAPL', days: int = 120, start_price: float = 150.0):
    """Erzeugt realistische OHLCV-Kursdaten mit Trends und Volatilität."""
    np.random.seed(hash(symbol) % 2**31)
    
    dates = pd.date_range(end=datetime.now(), periods=days, freq='B')
    
    # Random Walk mit Drift
    returns = np.random.normal(0.0005, 0.015, days)
    prices = start_price * np.exp(np.cumsum(returns))
    
    # OHLCV
    data = []
    for i, date in enumerate(dates):
        close = prices[i]
        daily_range = close * np.random.uniform(0.005, 0.03)
        open_price = close - daily_range * np.random.uniform(-0.5, 0.5)
        high = max(open_price, close) + daily_range * np.random.uniform(0, 0.5)
        low = min(open_price, close) - daily_range * np.random.uniform(0, 0.5)
        volume = int(np.random.uniform(50_000_000, 150_000_000))
        
        data.append({
            'date': date,
            'open': round(open_price, 2),
            'high': round(high, 2),
            'low': round(low, 2),
            'close': round(close, 2),
            'volume': volume,
        })
    
    return pd.DataFrame(data)

# Daten für mehrere Symbole generieren
SYMBOLS = ['AAPL', 'MSFT', 'GOOGL']
stock_data = {}
for sym in SYMBOLS:
    stock_data[sym] = generate_stock_data(sym, days=120, start_price={'AAPL': 150, 'MSFT': 350, 'GOOGL': 130}[sym])
    print(f'✅ {sym}: {len(stock_data[sym])} Tage geladen')

# Vorschau AAPL
stock_data['AAPL'].head(10)

## 3. Technische Indikatoren berechnen

Berechnung von Moving Averages (MA5, MA10, MA20, MA60), MACD und RSI.

In [ ]:
def calculate_indicators(df: pd.DataFrame) -> pd.DataFrame:
    """Berechnet alle technischen Indikatoren."""
    df = df.copy()
    
    # Moving Averages
    df['MA5'] = df['close'].rolling(window=5).mean()
    df['MA10'] = df['close'].rolling(window=10).mean()
    df['MA20'] = df['close'].rolling(window=20).mean()
    df['MA60'] = df['close'].rolling(window=60).mean()
    
    # MACD (12, 26, 9)
    ema12 = df['close'].ewm(span=12, adjust=False).mean()
    ema26 = df['close'].ewm(span=26, adjust=False).mean()
    df['MACD_DIF'] = ema12 - ema26
    df['MACD_DEA'] = df['MACD_DIF'].ewm(span=9, adjust=False).mean()
    df['MACD_BAR'] = (df['MACD_DIF'] - df['MACD_DEA']) * 2
    
    # RSI (14)
    delta = df['close'].diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    avg_gain = gain.ewm(alpha=1/14, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/14, adjust=False).mean()
    rs = avg_gain / avg_loss
    df['RSI'] = 100 - (100 / (1 + rs))
    
    # Volumen-Indikator (Volumen / 5-Tage-Durchschnitt)
    df['VOL_RATIO'] = df['volume'] / df['volume'].rolling(5).mean()
    
    return df

# Indikatoren für AAPL berechnen
df_aapl = calculate_indicators(stock_data['AAPL'])
print('📊 Indikatoren berechnet:')
print(f'  MA5: {df_aapl["MA5"].iloc[-1]:.2f}')
print(f'  MA10: {df_aapl["MA10"].iloc[-1]:.2f}')
print(f'  MA20: {df_aapl["MA20"].iloc[-1]:.2f}')
print(f'  MACD DIF: {df_aapl["MACD_DIF"].iloc[-1]:.4f}')
print(f'  RSI: {df_aapl["RSI"].iloc[-1]:.1f}')

df_aapl[['date', 'close', 'MA5', 'MA10', 'MA20', 'MACD_DIF', 'RSI']].tail(10)

## 4. Trend-Analyse

Bestimmung des Trend-Status basierend auf MA5 > MA10 > MA20.

In [ ]:
def analyze_trend(df: pd.DataFrame) -> dict:
    """Analysiert den aktuellen Trend-Status."""
    latest = df.iloc[-1]
    ma5, ma10, ma20 = latest['MA5'], latest['MA10'], latest['MA20']
    price = latest['close']
    
    # Trend-Status
    if ma5 > ma10 > ma20:
        trend = '🟢 Starker Aufwärtstrend (MA5 > MA10 > MA20)'
        strength = 80
    elif ma5 > ma10:
        trend = '🟡 Schwacher Aufwärtstrend (MA5 > MA10, aber MA10 ≤ MA20)'
        strength = 55
    elif ma5 < ma10 < ma20:
        trend = '🔴 Starker Abwärtstrend (MA5 < MA10 < MA20)'
        strength = 20
    elif ma5 < ma10:
        trend = '🟠 Schwacher Abwärtstrend (MA5 < MA10, aber MA10 ≥ MA20)'
        strength = 40
    else:
        trend = '⚪ Seitwärts / Konsolidierung'
        strength = 50
    
    # Bias (Abweichung von MA5)
    bias_ma5 = (price - ma5) / ma5 * 100 if ma5 > 0 else 0
    
    # RSI-Status
    rsi = latest['RSI']
    if rsi > 70:
        rsi_status = '🔥 Überkauft'
    elif rsi < 30:
        rsi_status = '❄️ Überverkauft'
    elif 40 <= rsi <= 60:
        rsi_status = '⚖️ Neutral'
    else:
        rsi_status = '📊 Normal'
    
    # MACD-Signal
    macd_dif = latest['MACD_DIF']
    macd_dea = latest['MACD_DEA']
    if macd_dif > macd_dea and macd_dif > 0:
        macd_signal = '✅ Bullish (DIF > DEA, über Null)'
    elif macd_dif > macd_dea:
        macd_signal = '↗️ Leicht bullish (DIF > DEA)'
    elif macd_dif < macd_dea and macd_dif < 0:
        macd_signal = '❌ Bearish (DIF < DEA, unter Null)'
    else:
        macd_signal = '↘️ Leicht bearish (DIF < DEA)'
    
    return {
        'Trend': trend,
        'Trend-Stärke': f'{strength}/100',
        'Preis': f'{price:.2f}',
        'MA5': f'{ma5:.2f}',
        'MA10': f'{ma10:.2f}',
        'MA20': f'{ma20:.2f}',
        'Bias MA5': f'{bias_ma5:.2f}%',
        'RSI': f'{rsi:.1f} ({rsi_status})',
        'MACD': macd_signal,
        'Volumen-Ratio': f'{latest["VOL_RATIO"]:.2f}x',
    }

# Analyse für alle Symbole
for sym in SYMBOLS:
    df = calculate_indicators(stock_data[sym])
    result = analyze_trend(df)
    print(f'\n📊 {sym}:')
    for key, value in result.items():
        print(f'  {key}: {value}')

## 5. Kurs-Chart mit Indikatoren

Visualisierung: Candlestick-ähnlicher Chart mit MA-Linien, MACD und RSI.

In [ ]:
def plot_stock_analysis(df: pd.DataFrame, symbol: str):
    """Erstellt einen umfassenden Analyse-Chart."""
    df = calculate_indicators(df)
    
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(16, 12),
        gridspec_kw={'height_ratios': [3, 1, 1]}, sharex=True)
    
    # ── Chart 1: Kurs + MAs ──────────────────────────────
    ax1.plot(df['date'], df['close'], color='#1a1a2e', linewidth=1.5, label='Close', alpha=0.8)
    ax1.plot(df['date'], df['MA5'], color='#e74c3c', linewidth=1, label='MA5', alpha=0.7)
    ax1.plot(df['date'], df['MA10'], color='#f39c12', linewidth=1, label='MA10', alpha=0.7)
    ax1.plot(df['date'], df['MA20'], color='#3498db', linewidth=1, label='MA20', alpha=0.7)
    ax1.plot(df['date'], df['MA60'], color='#9b59b6', linewidth=1, label='MA60', alpha=0.5, linestyle='--')
    
    # Fülle Bereich zwischen MA5 und MA20
    ax1.fill_between(df['date'], df['MA5'], df['MA20'],
                     where=df['MA5'] > df['MA20'], color='green', alpha=0.05)
    ax1.fill_between(df['date'], df['MA5'], df['MA20'],
                     where=df['MA5'] <= df['MA20'], color='red', alpha=0.05)
    
    ax1.set_ylabel('Preis (USD)', fontsize=12)
    ax1.set_title(f'{symbol} — Technische Analyse', fontsize=16, fontweight='bold')
    ax1.legend(loc='upper left', fontsize=9)
    ax1.grid(True, alpha=0.3)
    
    # ── Chart 2: MACD ────────────────────────────────────
    colors_macd = ['#2ecc71' if v >= 0 else '#e74c3c' for v in df['MACD_BAR'].fillna(0)]
    ax2.bar(df['date'], df['MACD_BAR'], color=colors_macd, width=0.8, alpha=0.7)
    ax2.plot(df['date'], df['MACD_DIF'], color='#3498db', linewidth=1, label='DIF')
    ax2.plot(df['date'], df['MACD_DEA'], color='#e67e22', linewidth=1, label='DEA')
    ax2.axhline(y=0, color='gray', linestyle='-', linewidth=0.5)
    ax2.set_ylabel('MACD', fontsize=12)
    ax2.legend(loc='upper left', fontsize=9)
    ax2.grid(True, alpha=0.3)
    
    # ── Chart 3: RSI ────────────────────────────────────
    ax3.plot(df['date'], df['RSI'], color='#8e44ad', linewidth=1.5, label='RSI (14)')
    ax3.axhline(y=70, color='red', linestyle='--', linewidth=0.8, alpha=0.6, label='Überkauft (70)')
    ax3.axhline(y=30, color='green', linestyle='--', linewidth=0.8, alpha=0.6, label='Überverkauft (30)')
    ax3.axhline(y=50, color='gray', linestyle=':', linewidth=0.5)
    ax3.fill_between(df['date'], 70, df['RSI'], where=df['RSI'] > 70, color='red', alpha=0.1)
    ax3.fill_between(df['date'], 30, df['RSI'], where=df['RSI'] < 30, color='green', alpha=0.1)
    ax3.set_ylabel('RSI', fontsize=12)
    ax3.set_xlabel('Datum', fontsize=12)
    ax3.set_ylim(0, 100)
    ax3.legend(loc='upper left', fontsize=9)
    ax3.grid(True, alpha=0.3)
    
    # Formatierung
    ax3.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m.%y'))
    ax3.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# Chart für AAPL
plot_stock_analysis(stock_data['AAPL'], 'AAPL')

## 6. Kauf-/Verkaufssignale

Generierung von Trading-Signalen basierend auf Trend, MACD und RSI.

In [ ]:
def generate_signal(df: pd.DataFrame) -> dict:
    """Generiert ein Trading-Signal mit Begründung."""
    df = calculate_indicators(df)
    latest = df.iloc[-1]
    prev = df.iloc[-2]
    
    score = 0
    reasons = []
    risks = []
    
    # 1. Trend-Check (max 40 Punkte)
    if latest['MA5'] > latest['MA10'] > latest['MA20']:
        score += 40
        reasons.append('✅ MA5 > MA10 > MA20 — Starker Aufwärtstrend')
    elif latest['MA5'] > latest['MA10']:
        score += 20
        reasons.append('⚠️ MA5 > MA10, aber MA10 ≤ MA20 — Schwacher Trend')
    else:
        risks.append('❌ Keine bullishe MA-Struktur')
    
    # 2. MACD-Check (max 25 Punkte)
    if latest['MACD_DIF'] > latest['MACD_DEA']:
        score += 15
        reasons.append('✅ MACD: DIF > DEA (bullish)')
        if latest['MACD_DIF'] > 0:
            score += 10
            reasons.append('✅ MACD über Nulllinie')
    else:
        risks.append('❌ MACD: DIF < DEA (bearish)')
    
    # 3. RSI-Check (max 20 Punkte)
    rsi = latest['RSI']
    if 40 <= rsi <= 60:
        score += 20
        reasons.append(f'✅ RSI {rsi:.1f} — Neutraler Bereich')
    elif 30 <= rsi < 40:
        score += 15
        reasons.append(f'⚠️ RSI {rsi:.1f} — Leicht überverkauft, mögliche Erholung')
    elif rsi > 70:
        risks.append(f'⚠️ RSI {rsi:.1f} — Überkauft!')
    elif rsi < 30:
        score += 10
        reasons.append(f'💡 RSI {rsi:.1f} — Stark überverkauft, Rebound möglich')
    
    # 4. Bias-Check (max 15 Punkte)
    bias = (latest['close'] - latest['MA5']) / latest['MA5'] * 100
    if -2 <= bias <= 2:
        score += 15
        reasons.append(f'✅ Preis nahe MA5 (Bias: {bias:.1f}%) — Guter Einstiegspunkt')
    elif bias > 5:
        risks.append(f'⚠️ Preis {bias:.1f}% über MA5 — Nicht nachkaufen!')
    
    # Signal
    if score >= 70:
        signal = '🟢 STRONG BUY'
    elif score >= 50:
        signal = '🟡 BUY'
    elif score >= 30:
        signal = '⚪ HOLD'
    elif score >= 15:
        signal = '🟠 SELL'
    else:
        signal = '🔴 STRONG SELL'
    
    return {
        'Signal': signal,
        'Score': f'{score}/100',
        'Gründe': reasons,
        'Risiken': risks,
    }

# Signale für alle Symbole
for sym in SYMBOLS:
    result = generate_signal(stock_data[sym])
    print(f'\n🎯 {sym}:')
    print(f'  Signal: {result["Signal"]} (Score: {result["Score"]})')
    for r in result['Gründe']:
        print(f'    {r}')
    for r in result['Risiken']:
        print(f'    {r}')

## 7. Multi-Asset-Vergleich

Vergleich der Kursentwicklung mehrerer Aktien (normalisiert).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Normalisierte Kursentwicklung
for sym in SYMBOLS:
    df = stock_data[sym]
    norm_price = df['close'] / df['close'].iloc[0] * 100
    ax1.plot(df['date'], norm_price, linewidth=2, label=sym)

ax1.set_title('Normalisierte Kursentwicklung (Basis = 100)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Performance (%)', fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.axhline(y=100, color='gray', linestyle='--', linewidth=0.5)

# Rendite-Statistik
returns_data = []
for sym in SYMBOLS:
    df = stock_data[sym]
    ret = (df['close'].iloc[-1] / df['close'].iloc[0] - 1) * 100
    volatility = df['close'].pct_change().std() * np.sqrt(252) * 100
    returns_data.append({'Symbol': sym, 'Rendite (%)': f'{ret:.1f}', 'Volatilität (%)': f'{volatility:.1f}'})

ax2.axis('tight')
ax2.axis('off')
table = ax2.table(cellText=[[d['Symbol'], d['Rendite (%)'], d['Volatilität (%)']] for d in returns_data],
                  colLabels=['Symbol', 'Rendite (%)', 'Volatilität (%, ann.)'],
                  cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.8)
ax2.set_title('Performance-Übersicht', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

## 8. Zusammenfassung

| Komponente | Beschreibung |
|---|---|
| 📡 **Daten** | OHLCV-Kursdaten (Demo oder Alpha Vantage) |
| 📊 **Indikatoren** | MA5/MA10/MA20/MA60, MACD, RSI, Volumen-Ratio |
| 🎯 **Trend** | 7 Trend-Status-Level (Strong Bull → Strong Bear) |
| 🔔 **Signale** | Strong Buy / Buy / Hold / Sell / Strong Sell |
| 📈 **Charts** | Kurs+MA, MACD-Balken, RSI mit Overbought/Oversold |

---

**Nächste Schritte:**
- Echte Kursdaten mit Alpha Vantage API laden
- LLM-Agenten für Reasoning einsetzen (`src/services/intelligence_service.py`)
- Streamlit-App starten: `streamlit run src/webui_frontend.py`
- W&B-Tracking für Experimente aktivieren